# Tool de ejecución de código

El modelo multiplica mal números grandes, pero escribe código bastante bien.
Así que le damos un tool que recibe código Python, lo ejecuta y devuelve lo que
ese código imprime.

El ciclo es el mismo de siempre:

1. Mandas la pregunta junto con la lista `tools`.
2. El modelo contesta con `tool_calls` en lugar de texto.
3. Ejecutas el código que pidió y devuelves la salida en un turno `role: "tool"`.
4. Llamas otra vez y ahora sí responde en palabras.

**Cuidado:** este notebook ejecuta literalmente lo que el modelo escriba.
Córrelo solo en tu máquina y revisa antes qué pidió: por eso imprimimos el
código antes de ejecutarlo.

**Necesitas una API key** en `.env` con el nombre `OPENROUTER_API_KEY`.

In [ ]:
import os
import io
import json
import contextlib

import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY"

## El tool

`ejecutar_python` corre el código y captura lo que se imprime con `print`.
Debajo, el JSON Schema que le describe la función al modelo: cómo se llama, para
qué sirve y qué parámetro recibe.

In [ ]:
def ejecutar_python(codigo):
    salida = io.StringIO()
    with contextlib.redirect_stdout(salida):
        exec(codigo)
    return salida.getvalue()


tools = [
    {
        "type": "function",
        "function": {
            "name": "ejecutar_python",
            "description": "Ejecuta código Python y devuelve lo que ese código imprime con print.",
            "parameters": {
                "type": "object",
                "properties": {
                    "codigo": {
                        "type": "string",
                        "description": "Código Python a ejecutar. Debe imprimir el resultado con print.",
                    },
                },
                "required": ["codigo"],
            },
        },
    },
]

## El llamado al LLM

La misma llamada de siempre a `POST /chat/completions`, con un campo nuevo:
`tools`. Si el modelo decide usar la herramienta, en vez de `content` viene una
lista `tool_calls` con el nombre de la función y sus argumentos en JSON.

In [ ]:
messages = [
    {"role": "user", "content": "¿cuánto es 4573 × 892?"},
]

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": messages,
        "tools": tools,
    },
    timeout=60,
)
r.raise_for_status()

respuesta = r.json()["choices"][0]["message"]
print(json.dumps(respuesta, indent=2, ensure_ascii=False))

## Revisa antes de ejecutar

Primero miramos qué código quiere correr el modelo. Léelo: si pide borrar
archivos o leer algo tuyo, no sigas a la celda siguiente.

In [ ]:
llamada = respuesta["tool_calls"][0]
codigo = json.loads(llamada["function"]["arguments"])["codigo"]

print(llamada["function"]["name"])
print(codigo)

## El uso del tool

Ejecutamos el código, agregamos el turno `role: "tool"` con la salida y el
`tool_call_id` que llegó, y hacemos la segunda llamada. Ahí el modelo ya ve el
resultado y contesta en palabras.

In [ ]:
resultado = ejecutar_python(codigo)
print(resultado)

messages.append(respuesta)
messages.append(
    {
        "role": "tool",
        "tool_call_id": llamada["id"],
        "content": resultado,
    }
)

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": messages,
        "tools": tools,
    },
    timeout=60,
)
r.raise_for_status()

print(r.json()["choices"][0]["message"]["content"])

## Otra tarea

La misma vuelta completa, ahora con una tarea de listas y texto. El modelo puede
pedir varias ejecuciones, así que recorremos todos los `tool_calls` e imprimimos
cada código antes de correrlo.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Ordena de mayor a menor la lista [8, 3, 91, 12, 45] y dime cuántas palabras tiene la frase 'el gato duerme sobre el techo caliente'",
    },
]

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": messages,
        "tools": tools,
    },
    timeout=60,
)
r.raise_for_status()

respuesta = r.json()["choices"][0]["message"]
messages.append(respuesta)

for llamada in respuesta["tool_calls"]:
    codigo = json.loads(llamada["function"]["arguments"])["codigo"]
    print(codigo)
    messages.append(
        {
            "role": "tool",
            "tool_call_id": llamada["id"],
            "content": ejecutar_python(codigo),
        }
    )

r = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": messages,
        "tools": tools,
    },
    timeout=60,
)
r.raise_for_status()

print(r.json()["choices"][0]["message"]["content"])